# Chapter 5a — Regression
**MADT6004 · Brew Lab BKK case**

Regression goes beyond correlation: it tells you **how much** of the outcome each variable explains, while controlling for the others.

You will:
1. Fit a simple linear regression (one predictor)
2. Fit a multiple regression (several predictors + categorical control)
3. Read the coefficients and R²


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
import statsmodels.formula.api as smf

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Branch-day panel with predictors

In [ ]:
panel = pd.read_sql("""
SELECT date(t.datetime) AS d, t.branch_id,
       SUM(t.total) AS revenue,
       COUNT(*)     AS orders
FROM transactions t
GROUP BY t.branch_id, date(t.datetime)
""", conn)
br = pd.read_sql("""
SELECT branch_id, district, seats, size_sqm, has_drive_thru, base_traffic
FROM branches
""", conn)
panel = panel.merge(br, on="branch_id")
panel["d"] = pd.to_datetime(panel["d"])
panel["is_weekend"] = (panel["d"].dt.dayofweek >= 5).astype(int)
print(panel.head())


## 3. Simple linear regression
Predict revenue from base traffic alone.

In [ ]:
m1 = smf.ols("revenue ~ base_traffic", data=panel).fit()
print(m1.summary().tables[1])
print(f"\nR-squared: {m1.rsquared:.3f}")


## 4. Multiple regression with controls
Add seats, size, drive-thru, weekend, and district fixed effects.

In [ ]:
m2 = smf.ols(
    "revenue ~ base_traffic + seats + size_sqm + has_drive_thru + is_weekend + C(district)",
    data=panel
).fit()
print(m2.summary().tables[1])
print(f"\nR-squared: {m2.rsquared:.3f}  (vs simple: {m1.rsquared:.3f})")


## 5. Plot residuals
A good model has residuals scattered around zero with no pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(m2.fittedvalues, m2.resid, alpha=0.25, color="#0891B2")
ax.axhline(0, color="red", lw=0.7)
ax.set_xlabel("Fitted revenue"); ax.set_ylabel("Residual")
ax.set_title("Residuals vs fitted")
plt.tight_layout(); plt.show()


## Discussion prompts
1. Which predictor's coefficient surprised you most? Why?
2. R² rose when you added more predictors — but is that always a good sign?
3. The residual plot shows the model's leftover error. What patterns would worry you?
